In [1]:
# Install Library

!pip install transformers torch scikit-learn

In [2]:
# Upload Dataset

from google.colab import files
uploaded = files.upload()

Saving IMDB Dataset.csv to IMDB Dataset.csv


In [3]:
# Load Dataset

import pandas as pd

df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
# Explore Data

df.info()
df['sentiment'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


,count
sentiment,
positive,25000
negative,25000


In [5]:
# Preprocessing

import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text

df['review'] = df['review'].apply(clean_text)

In [6]:
df['label'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

In [7]:
df = df.sample(200)

In [8]:
# Data Splitting

from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['review'], df['label'], test_size=0.2, random_state=42
)

print("Split done")

Split done


In [9]:
# Tokenization

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(list(train_texts), truncation=True, padding=True)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True)

print("Tokenization done")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenization done


In [10]:
# Dataset Class

import torch

class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = IMDbDataset(train_encodings, train_labels)
test_dataset = IMDbDataset(test_encodings, test_labels)

print("Dataset ready")

Dataset ready


In [11]:
# Load BERT Model

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

print("Model loaded")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded


In [12]:
# Metrics

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

In [13]:
# Training Arguments

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=1,
    report_to="none"
)


In [14]:
# Trainer

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("Trainer ready")

Trainer ready


In [15]:
# Train Model

trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=20, training_loss=0.714493703842163, metrics={'train_runtime': 25.0503, 'train_samples_per_second': 6.387, 'train_steps_per_second': 0.798, 'total_flos': 42097768857600.0, 'train_loss': 0.714493703842163, 'epoch': 1.0})

In [27]:
# Evaluate Model

result = trainer.evaluate()
print(result)

{'eval_loss': 0.6520280838012695, 'eval_accuracy': 0.625, 'eval_f1': 0.0, 'eval_precision': 0.0, 'eval_recall': 0.0, 'eval_runtime': 1.6026, 'eval_samples_per_second': 24.959, 'eval_steps_per_second': 3.12, 'epoch': 1.0}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [17]:
# Freeze BERT Layers

from transformers import AutoModelForSequenceClassification

model_frozen = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [18]:
for param in model_frozen.bert.parameters():
    param.requires_grad = False

In [19]:
trainer_frozen = Trainer(
    model=model_frozen,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer_frozen.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=20, training_loss=0.7062437534332275, metrics={'train_runtime': 9.6411, 'train_samples_per_second': 16.596, 'train_steps_per_second': 2.074, 'total_flos': 42097768857600.0, 'train_loss': 0.7062437534332275, 'epoch': 1.0})

In [20]:
result_frozen = trainer_frozen.evaluate()
print(result_frozen)

{'eval_loss': 0.7045881152153015, 'eval_accuracy': 0.375, 'eval_f1': 0.4897959183673469, 'eval_precision': 0.35294117647058826, 'eval_recall': 0.8, 'eval_runtime': 1.6223, 'eval_samples_per_second': 24.657, 'eval_steps_per_second': 3.082, 'epoch': 1.0}


In [21]:
# Fine-tuned Last 2 Layers

model_last2 = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [22]:
for param in model_last2.bert.parameters():
    param.requires_grad = False

In [23]:
for param in model_last2.bert.encoder.layer[-2:].parameters():
    param.requires_grad = True

In [24]:
trainer_last2 = Trainer(
    model=model_last2,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer_last2.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=20, training_loss=0.7039031505584716, metrics={'train_runtime': 11.2992, 'train_samples_per_second': 14.16, 'train_steps_per_second': 1.77, 'total_flos': 42097768857600.0, 'train_loss': 0.7039031505584716, 'epoch': 1.0})

In [25]:
result_last2 = trainer_last2.evaluate()
print(result_last2)

{'eval_loss': 0.6743811368942261, 'eval_accuracy': 0.65, 'eval_f1': 0.2222222222222222, 'eval_precision': 0.6666666666666666, 'eval_recall': 0.13333333333333333, 'eval_runtime': 1.5393, 'eval_samples_per_second': 25.986, 'eval_steps_per_second': 3.248, 'epoch': 1.0}


In [28]:
# Compare Result

import pandas as pd

results = pd.DataFrame({
    "Experiment": ["Normal BERT", "Frozen BERT", "Last 2 Layers"],
    "Accuracy": [
        result['eval_accuracy'],
        result_frozen['eval_accuracy'],
        result_last2['eval_accuracy']
    ]
})

print(results)

      Experiment  Accuracy
0    Normal BERT     0.625
1    Frozen BERT     0.375
2  Last 2 Layers     0.650


**Observations:**

1. The fully trained BERT model gave the highest accuracy.
2. Freezing BERT layers reduced training time but lowered accuracy.
3. Fine-tuning the last 2 layers improved performance compared to frozen BERT.
4. There is a trade-off between training time and model performance.

**Conclusion:**

In this project, BERT was used for sentiment analysis on the IMDB dataset.
Different experiments were performed including freezing layers and partial fine-tuning.

The results show that:
- Fully fine-tuned BERT provides the best accuracy.
- Freezing layers reduces computation but affects performance.
- Fine-tuning a few layers gives a good balance.

Thus, BERT is highly effective for text classification tasks.